# K-IFRS Agent End-to-End 평가

서브에이전트(retrieval-distiller) 도입 후 **최종 답변 품질**을 golden dataset 36문항으로 평가.

**지표:**
- **Cited Recall**: 최종 답변에서 expected_paragraphs가 인용된 비율
- **Standard Accuracy**: 정답 기준서가 인용에 포함되었는지
- **Latency**: 질문→답변 소요 시간

**비교 대상:** 기존 retriever-only 평가 (baseline Recall=0.540, StdAcc=0.889)

In [3]:
import json
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from eval.evaluate_agent import (
    extract_paragraph_citations,
    compute_agent_metrics,
    run_agent_evaluation,
    run_adhoc_query,
    _extract_final_answer,
)

GOLDEN = json.loads(Path("eval/golden_dataset.json").read_text())
print(f"Golden dataset: {len(GOLDEN)}문항")

Golden dataset: 36문항


## 1. 단건 테스트

전체 돌리기 전에 1문항으로 파이프라인이 정상 동작하는지 확인.

## 자유 질의

Golden dataset 없이 임의의 질문을 던져볼 때 사용. `run_adhoc_query(query)`는 문자열만 받는다.

In [1]:
from IPython.display import Markdown, display

display(Markdown(f"\n사용자 질문: \\n\n다음 주제와 관련된 K-IFRS 기준서 본문·적용지침·정의 문단을 검색하여 JSON 형식으로 반환하세요:\n\n1. 기한이익상실(covenant breach / acceleration clause) 조건 위반 시 유동/비유동 부채 분류\n2. 보고기간 후 채무 재협상(refinancing) 또는 유예(waiver)가 분류에 미치는 영향\n3. 장기 차입금이 유동부채로 재분류되는 조건\n4. 관련 공시 요구사항\n\n관련 기준서: K-IFRS 1001(재무제표 표시), K-IFRS 1010(보고기간후사건) 등을 중심으로 검색하세요.\n\n반환 형식:\n{{\n  \"synthesis\": \"핵심 내용 요약\",\n  \"chunks\": [\n    {{\n      \"standard_id\": \"K-IFRS 1001\",\n      \"paragraph\": \"문단 번호\",\n      \"original_text\": \"원문\",\n      \"summary\": \"요약\"\n    }}\n  ],\n  \"notes\": \"추가 참고사항\"\n}}\n"))


사용자 질문: \n
다음 주제와 관련된 K-IFRS 기준서 본문·적용지침·정의 문단을 검색하여 JSON 형식으로 반환하세요:

1. 기한이익상실(covenant breach / acceleration clause) 조건 위반 시 유동/비유동 부채 분류
2. 보고기간 후 채무 재협상(refinancing) 또는 유예(waiver)가 분류에 미치는 영향
3. 장기 차입금이 유동부채로 재분류되는 조건
4. 관련 공시 요구사항

관련 기준서: K-IFRS 1001(재무제표 표시), K-IFRS 1010(보고기간후사건) 등을 중심으로 검색하세요.

반환 형식:
{
  "synthesis": "핵심 내용 요약",
  "chunks": [
    {
      "standard_id": "K-IFRS 1001",
      "paragraph": "문단 번호",
      "original_text": "원문",
      "summary": "요약"
    }
  ],
  "notes": "추가 참고사항"
}


In [5]:
from IPython.display import Markdown, display

query = "무형자산을 개별적으로 식별할 기준에 관해서 설명해주세요"

result = run_adhoc_query(query)

print(f"Latency: {result['latency_sec']}s")
print(f"Cited: {result['cited_paragraphs']}\n")
display(Markdown(result["answer_text"]))

Latency: 54.04s
Cited: [('K-IFRS 1103', 'B33'), ('K-IFRS 1103', 'B31'), ('K-IFRS 1038', '21')]



## 무형자산의 개별 식별 기준

### 핵심 답변

무형자산이 "식별할 수 있다"고 인정받으려면, **① 분리 가능성 기준** 또는 **② 계약적·법적 기준** 중 하나를 충족해야 합니다. 이 두 기준 중 하나라도 충족되면 영업권과 분리하여 별도 인식합니다.

---

### 1. 무형자산의 기본 정의

K-IFRS 제1038호에 따르면:

> *"무형자산을 '물리적 실체는 없지만 **식별할 수 있는** 비화폐성자산'으로 정의하고 있다."*

'식별 가능성'은 무형자산의 정의 자체에 내재된 핵심 요건입니다. 단순히 영업권의 일부로만 존재하는 자산은 무형자산으로 인식하지 않습니다.

---

### 2. 두 가지 식별 기준

#### ① 분리 가능성 기준 (Separability Criterion)

K-IFRS 제1103호 문단 B33:

> *"분리 가능성 기준은 취득한 무형자산이 피취득자에게서 분리되거나 분할될 수 있고, 개별적으로 또는 관련된 계약, 식별할 수 있는 자산이나 부채와 함께 **매각·이전·라이선스·임대·교환**을 할 수 있음을 의미한다. 취득자가 매각, 라이선스, 교환을 할 의도가 없더라도, 취득자가 매각, 라이선스, 그 밖의 기타 가치 있는 것과 교환할 수 있는 무형자산은 분리 가능성 기준을 충족한다. 취득한 무형자산은 바로 그 형태의 자산이나 비슷한 형태의 자산과의 교환거래에 대한 증거가 있는 경우에 그러한 교환거래가 드물고 취득자가 그 거래와 관련이 있는지와 무관하게 분리 가능성 기준을 충족한다."*

**핵심 포인트:**
- 기업의 실제 **매각 의도가 없어도** 분리 가능성만 있으면 충족
- 시장에서 **유사 자산의 교환거래 증거**가 있으면 분리 가능성 인정
- 해당 자산만 단독으로 분리되지 않더라도, **관련 계약·자산·부채와 함께** 분리 가능하면 충족

---

#### ② 계약적·법적 기준 (Contractual/Legal Criterion)

K-IFRS 제1103호 문단 B31, B32:

> *"계약적·법적 기준을 충족하는 무형자산은 **피취득자에게서 또는 그 밖의 권리와 의무에서 이전하거나 분리할 수 없더라도** 식별할 수 있다."*

구체적인 사례:
- **원자력 발전소 운영 라이선스**: 발전소에서 물리적으로 분리·매각 불가능해도, 법적 권리로 존재하므로 영업권과 분리하여 인식
- **기술특허권 및 관련 라이선스 약정**: 서로 분리하여 실무적으로 매각하거나 교환할 수 없더라도, 각각 계약적·법적 기준을 충족

**핵심 포인트:**
- 분리 가능성이 없더라도 **계약상 또는 법률상 권리에서 발생**하면 별도 식별 가능
- 분리 가능성 기준보다 더 넓은 범위를 커버하는 기준

---

### 3. 개별 인식 요건 (식별 이후)

K-IFRS 제1038호 문단 21에 따라, 식별된 무형자산을 실제로 재무제표에 인식하려면 추가로 다음 두 조건을 모두 충족해야 합니다:

> *"⑴ 자산에서 발생하는 미래경제적효익이 기업에 유입될 **가능성이 높다.**"*
> *"⑵ 자산의 원가를 **신뢰성 있게 측정**할 수 있다."*

---

### 요약 정리

| 기준 | 설명 | 분리 이전 가능성 여부 |
|------|------|----------------------|
| **분리 가능성** | 매각·이전·라이선스·임대·교환 가능 | 필요 (단독 또는 묶음) |
| **계약적·법적** | 계약상·법률상 권리에서 발생 | 불필요 (분리 불가해도 인정) |

두 기준 중 하나만 충족하면 무형자산으로 식별되며, 식별된 이후 경제적효익 유입 가능성 및 원가 측정 가능성을 추가로 충족해야 최종 인식됩니다.

In [11]:
# 첫 번째 문항으로 smoke test
item = GOLDEN[20]
print(f"Query: {item['query']}")
print(f"Expected: {item['expected_standard']} 문단 {item['expected_paragraphs']}")

result = run_agent_evaluation(item)
metrics = compute_agent_metrics(result)

# print(f"\n--- 결과 ---")
# print(f"Latency: {result['latency_sec']}s")
# print(f"Cited paragraphs: {result['cited_paragraphs']}")
# print(f"Std hit: {metrics['std_hit']}, Cited recall: {metrics['cited_recall']}")
# print(f"\n--- 답변 (앞 500자) ---")
# print(result["answer_text"][:500])

from IPython.display import Markdown, display
display(Markdown(result['answer_text']))

Query: 재고자산의 저가법 평가
Expected: K-IFRS 1002 문단 ['28', '29']


IE 없음: K-IFRS 1002, query=재고자산 순실현가능가치 저가법 평가손실 사례


---

## 재고자산의 저가법(低價法) 평가

---

### 1. 핵심 원칙

재고자산은 **취득원가와 순실현가능가치(NRV) 중 낮은 금액**으로 측정합니다. 이는 자산이 원가를 회수할 수 없는 경우 과대평가되어서는 안 된다는 원칙에 근거합니다.

> **K-IFRS 1002 문단 9**
> "재고자산은 취득원가와 순실현가능가치 중 낮은 금액으로 측정한다."

---

### 2. 저가법 적용 사유

다음과 같은 상황에서 재고자산의 원가가 회수되지 않을 수 있습니다.

> **K-IFRS 1002 문단 28**
> 재고자산의 취득원가가 순실현가능가치보다 높을 수 있는 경우:
> - **손상·진부화·판매가격 하락**: 재고자산이 훼손된 경우, 전부 또는 일부가 진부화된 경우, 판매가격이 하락한 경우
> - **완성·판매 비용 증가**: 완성하는 데 필요한 원가 또는 판매하는 데 필요한 원가가 상승한 경우

---

### 3. 순실현가능가치(NRV) 측정

> **K-IFRS 1002 문단 30**
> 순실현가능가치 추정은 재고자산의 **처분으로부터 실현할 것으로 기대하는 금액에 기초**하며, 보고기간 후 사건이 보고기간 말 현재의 상황을 확인해 주는 경우 해당 가격 또는 원가 변동을 고려합니다.

| 구분 | NRV 산정 방법 |
|---|---|
| 일반 재고자산 | 예상 판매가격 − 예상 판매비용 |
| 원재료 | 완성품의 NRV ≥ 원가 → 감액 불필요. 완성품의 NRV < 원가 → 현행대체원가로 측정 |
| 확정판매계약용 재고 | 계약가격 기준으로 NRV 산정 |

---

### 4. 항목별 적용 방법

> **K-IFRS 1002 문단 29**
> 재고자산 감액은 **항목별**로 수행하는 것이 원칙입니다.
> 동일한 목적이나 용도로 생산·판매되며 동일 지역에서 판매되는 경우처럼, 항목별 적용이 실무적으로 불가능한 경우에는 **유사한 항목끼리 집합하여(집합법)** 적용할 수 있습니다.
> 단, **전체 재고자산 범주나 특정 사업부문 단위**로 감액하는 것은 적절하지 않습니다.

---

### 5. 평가손실 인식 및 환입

**① 평가손실 인식**

> **K-IFRS 1002 문단 34**
> NRV로의 감액분(재고자산평가손실)은 감액이 발생한 기간에 **비용(매출원가)으로 인식**합니다.

**② 환입 요건**

> **K-IFRS 1002 문단 33**
> 재고자산을 감액하여 NRV로 장부금액을 낮춘 경우, **순실현가능가치가 후속적으로 상승하면 환입**합니다. 환입액은 환입이 발생한 기간의 **비용(매출원가)을 감소**시킵니다.
> 단, 환입 후 장부금액은 **원래의 취득원가를 초과할 수 없습니다.**

---

### 요약 흐름도

```
취득원가  vs  순실현가능가치(NRV)
         ↓
   낮은 금액으로 측정
         ↓
취득원가 > NRV  →  차액을 재고자산평가손실로 비용 인식
         ↓
후속 기간에 NRV 회복  →  기존 평가손실 범위 내에서 환입 (취득원가 한도)
```

---

**적용 기준서**: K-IFRS 제1002호 '재고자산' (문단 9, 28~30, 33~34)

## 2. 전체 36문항 평가

중간 결과를 셀마다 출력하고, 실패해도 나머지를 계속 진행.
결과는 `eval/results/subagent.json`에 자동 저장.

In [ ]:
results = []
total_time = 0.0

for i, item in enumerate(GOLDEN):
    try:
        result = run_agent_evaluation(item)
    except Exception as e:
        print(f"  [ERR] {item['id']}: {e}")
        result = {
            "id": item["id"],
            "query": item["query"],
            "expected_standard": item["expected_standard"],
            "expected_paragraphs": item["expected_paragraphs"],
            "answer_text": "",
            "cited_paragraphs": [],
            "latency_sec": 0.0,
            "error": str(e),
        }

    metrics = compute_agent_metrics(result)
    result["metrics"] = metrics
    total_time += result["latency_sec"]
    results.append(result)

    status = "HIT" if metrics["std_hit"] else "MISS"
    print(
        f"[{i+1:2d}/36] [{status:>4}] {item['id']}: "
        f"cited_recall={metrics['cited_recall']:.2f} "
        f"n_cit={metrics['n_citations']} "
        f"({result['latency_sec']:.1f}s)"
    )

print(f"\n전체 소요 시간: {total_time:.1f}s")

## 3. 집계 및 저장

In [ ]:
avg_recall = sum(r["metrics"]["cited_recall"] for r in results) / len(results)
std_acc = sum(r["metrics"]["std_hit"] for r in results) / len(results)
avg_latency = total_time / len(results)

summary = {
    "config": "subagent",
    "n_queries": len(results),
    "avg_cited_recall": round(avg_recall, 3),
    "std_accuracy": round(std_acc, 3),
    "avg_latency_sec": round(avg_latency, 2),
    "total_time_sec": round(total_time, 1),
    "results": results,
}

out_path = Path("eval/results/subagent.json")
out_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str))

print(f"{'='*60}")
print(f"Config: subagent")
print(f"Queries: {len(results)}")
print(f"Avg Cited Recall: {avg_recall:.3f}")
print(f"Std Accuracy: {std_acc:.3f}")
print(f"Avg Latency: {avg_latency:.2f}s")
print(f"Saved: {out_path}")

## 4. 결과 분석

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "id": r["id"],
        "query": r["query"][:30],
        "expected_std": r["expected_standard"],
        "std_hit": r["metrics"]["std_hit"],
        "cited_recall": r["metrics"]["cited_recall"],
        "n_citations": r["metrics"]["n_citations"],
        "latency": r["latency_sec"],
        "error": r.get("error", ""),
    }
    for r in results
])

print("=== 전체 요약 ===")
print(df[["std_hit", "cited_recall", "n_citations", "latency"]].describe().round(3))

print("\n=== MISS 항목 ===")
display(df[df["std_hit"] == 0][["id", "query", "expected_std", "cited_recall", "n_citations"]])

In [ ]:
# 기준서별 성능
std_perf = df.groupby("expected_std").agg(
    n=("id", "count"),
    std_acc=("std_hit", "mean"),
    avg_recall=("cited_recall", "mean"),
    avg_latency=("latency", "mean"),
).round(3).sort_values("avg_recall")

print("=== 기준서별 성능 (recall 오름차순) ===")
display(std_perf)

## 5. Baseline(retriever-only) 비교

In [ ]:
# 기존 retriever-only baseline 결과 로드
baseline_path = Path("eval/results/reranker_kiwi.json")
if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text())
    print(f"{'지표':<25} {'Retriever-only':>15} {'Subagent E2E':>15}")
    print("-" * 57)
    print(f"{'Recall (retriever/cited)':<25} {baseline.get('avg_recall', 'N/A'):>15} {avg_recall:>15.3f}")
    print(f"{'Std Accuracy':<25} {baseline.get('std_accuracy', 'N/A'):>15} {std_acc:>15.3f}")
    print(f"{'Avg Latency (s)':<25} {baseline.get('avg_latency_sec', 'N/A'):>15} {avg_latency:>15.2f}")
    print("\n※ Retriever Recall = 검색 결과에 포함된 비율")
    print("※ Cited Recall = 최종 답변에서 실제 인용된 비율 (더 엄격)")
else:
    print("baseline 결과 파일 없음 — eval/evaluate.py baseline 먼저 실행 필요")

## 6. 개별 답변 확인

특정 문항의 답변 전문과 인용 현황을 확인.

In [ ]:
# 확인할 문항 ID 지정
TARGET_ID = "q001"

r = next((r for r in results if r["id"] == TARGET_ID), None)
if r:
    print(f"Query: {r['query']}")
    print(f"Expected: {r['expected_standard']} 문단 {r['expected_paragraphs']}")
    print(f"Cited: {r['cited_paragraphs']}")
    print(f"Metrics: {r['metrics']}")
    print(f"Latency: {r['latency_sec']}s")
    print(f"\n{'='*60}")
    print(r["answer_text"])
else:
    print(f"{TARGET_ID} not found")